In [1]:
!pip install catboost lightgbm -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install pygeohash


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 16.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
# ============================================================
# LIGHTGBM VERSION (CATBOOST STYLE) - EDITED
# ============================================================

import pandas as pd
import numpy as np
import pygeohash as pgh   # for lat/long decoding

from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ============================================================
# LOAD DATA
# ============================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

# ============================================================
# FEATURE ENGINEERING
# ============================================================

for df in [train, test]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%H:%M')
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute

    # Cyclical encoding
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

    df['is_weekend'] = (df['day'] % 7 >= 5).astype(int)

# ============================================================
# GEOHASH FEATURES
# ============================================================

for i in range(6):
    train[f'geo_{i}'] = train['geohash'].str[i]
    test[f'geo_{i}'] = test['geohash'].str[i]

freq = train['geohash'].value_counts()
train['geo_freq'] = train['geohash'].map(freq)
test['geo_freq'] = test['geohash'].map(freq).fillna(0)

# Decode lat/long from geohash
train['lat'] = train['geohash'].apply(lambda g: pgh.decode(g)[0])
train['lon'] = train['geohash'].apply(lambda g: pgh.decode(g)[1])
test['lat'] = test['geohash'].apply(lambda g: pgh.decode(g)[0])
test['lon'] = test['geohash'].apply(lambda g: pgh.decode(g)[1])

# ============================================================
# INTERACTION FEATURES
# ============================================================

for df in [train, test]:
    df['road_weather'] = df['RoadType'].astype(str) + "_" + df['Weather'].astype(str)
    df['geo_hour'] = df['geohash'].astype(str) + "_" + df['hour'].astype(str)
    df['weather_hour'] = df['Weather'].astype(str) + "_" + df['hour'].astype(str)

# ============================================================
# TARGET ENCODING (geo × hour)
# ============================================================

geo_hour_target = train.groupby(['geohash','hour'])['demand'].mean()
train['geo_hour_target'] = train.set_index(['geohash','hour']).index.map(geo_hour_target)
test['geo_hour_target'] = test.set_index(['geohash','hour']).index.map(geo_hour_target).fillna(train['demand'].mean())

# ============================================================
# LAG FEATURES (previous hour demand per geohash)
# ============================================================

train = train.sort_values(['geohash','timestamp'])
train['lag1'] = train.groupby('geohash')['demand'].shift(1)
train['lag1'] = train['lag1'].fillna(train['demand'].median())

test = test.sort_values(['geohash','timestamp'])
# The 'demand' column is not available in the test set. Directly fill 'lag1' with the median demand from the training set.
test['lag1'] = train['demand'].median()

# ============================================================
# PREPARE DATA
# ============================================================

y_log = np.log1p(train['demand'])
X = train.drop(['demand','timestamp','Index'], axis=1)
X_test = test.drop(['timestamp','Index'], axis=1)

# ============================================================
# MISSING VALUES (grouped imputation for weather/temperature)
# ============================================================

num_cols = X.select_dtypes(exclude='object').columns
for col in num_cols:
    X[col] = X[col].fillna(X.groupby('hour')[col].transform('median'))
    X_test[col] = X_test[col].fillna(X[col].median())

cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype("category")
    X_test[col] = X_test[col].fillna("Missing").astype("category")

# ============================================================
# TRAIN VALID SPLIT
# ============================================================

X_train, X_valid, y_train_log, y_valid_log = train_test_split(
    X, y_log, test_size=0.20, random_state=42
)

# ============================================================
# TUNED LIGHTGBM
# ============================================================

model = LGBMRegressor(
    objective='regression',
    n_estimators=8000,
    learning_rate=0.01,
    num_leaves=511,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=42,
    verbosity=-1
)

model.fit(
    X_train, y_train_log,
    eval_set=[(X_valid, y_valid_log)],
    eval_metric='rmse',
    callbacks=[
        __import__('lightgbm').early_stopping(500),
        __import__('lightgbm').log_evaluation(200)
    ]
)

# ============================================================
# VALIDATION SCORE
# ============================================================

val_pred_log = model.predict(X_valid)
val_pred = np.expm1(val_pred_log)

r2 = r2_score(np.expm1(y_valid_log), val_pred)
print("\nValidation R² =", r2)

score = max(0, 100*r2)
print("Competition Score =", score)

# ============================================================
# RETRAIN ON FULL DATA
# ============================================================

model.fit(X, y_log)

# ============================================================
# TEST PREDICTIONS
# ============================================================

final_pred_log = model.predict(X_test)
final_pred = np.expm1(final_pred_log)
final_pred[final_pred < 0] = 0

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_pred
})

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved")
print(submission.head())


FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'